# VNExpress Recommendation Benchmark

**Comprehensive ablation study** comparing CF, CB, and Hybrid approaches.

## Experiment Design
- **CF Models**: SimGCL, XSimGCL, LightGCL, MA-HCL, HetGNN, MA-HGN, Sim-MAHGN, NGCF
- **CB Models**: TF-IDF, VN-SBERT, BGE-M3
- **Hybrid Models**: SimGCL+VN-SBERT, XSimGCL+BGE-M3, CGRC+embeddings
- **Protocols**: Cold-Start, Full Ranking, LOO100
- **Graphs**: G1 (Bipartite), G2 (Heterogeneous), G3 (With Negatives)

## Step 1: Setup Environment

In [ ]:
# Clone repository and install dependencies
!git clone https://ghp_xxxxx@github.com/GadGadGad/DS300-Final-Project.git project
%cd project
!pip install -q torch torch_geometric pandas numpy scikit-learn tqdm sentence-transformers underthesea
print('Setup complete!')

## Step 2: Prepare Data

In [ ]:
# Copy data from Kaggle input
!mkdir -p data/raw data/processed checkpoints models results/ablation
!cp -r /kaggle/input/vnexpress-data/* data/raw/ 2>/dev/null || echo 'Using existing data'
!ls data/raw/

## Step 3: Generate Graphs (FRESH - No Leakage)

**IMPORTANT**: We regenerate all graphs from scratch using the leakage-fixed converter.
This ensures the message-passing graph only contains training edges.

In [ ]:
# Clean old graphs to force regeneration
!rm -rf data/processed/strict_g1 data/processed/strict_g2 data/processed/strict_g3
print('Cleaned old graphs. Regenerating...')

In [ ]:
# G1: Strict Bipartite (User-Article only) - LEAKAGE FIXED
!python src/data/convert_to_gnn.py \
    --graph-type user-article \
    --output data/processed/strict_g1 \
    --min-user-interactions 2 \
    --min-article-interactions 2

print('\n✅ G1 generated with train-only edges')

In [ ]:
# G2: Heterogeneous (+ Social edges) - LEAKAGE FIXED
!python src/data/convert_to_gnn.py \
    --graph-type hetero \
    --output data/processed/strict_g2 \
    --min-user-interactions 2 \
    --min-article-interactions 2

print('\n✅ G2 generated with train-only edges')

In [ ]:
# G3: With Negatives (for training) - LEAKAGE FIXED
!python src/data/convert_to_gnn.py \
    --graph-type with-negatives \
    --output data/processed/strict_g3 \
    --min-user-interactions 2 \
    --min-article-interactions 2 \
    --neg-ratio 1 \
    --neg-strategy random

print('\n✅ G3 generated with train-only edges')

## Step 4: Run Ablation Study

This will train all CF, CB, and Hybrid models across all graphs and protocols.

In [ ]:
# Run the comprehensive ablation script
!chmod +x scripts/run_ablation.sh
!./scripts/run_ablation.sh

## Step 5: Load and Analyze Results

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load all results
results = []
for f in os.listdir('results/ablation'):
    if f.endswith('.json'):
        with open(f'results/ablation/{f}') as file:
            data = json.load(file)
            # Parse filename
            parts = f.replace('.json', '').split('_')
            if parts[0] == 'cf':
                model_type = 'CF'
                model = parts[1]
                graph = parts[2]
                protocol = parts[3]
            elif parts[0] == 'cb':
                model_type = 'CB'
                model = parts[1]
                graph = parts[2]
                protocol = parts[3]
            else:  # hybrid
                model_type = 'Hybrid'
                model = f'{parts[1]}+{parts[2]}'
                graph = parts[3]
                protocol = parts[4]
            
            results.append({
                'Type': model_type,
                'Model': model,
                'Graph': graph,
                'Protocol': protocol,
                'Recall@10': data.get('recall@10', 0),
                'NDCG@10': data.get('ndcg@10', 0),
                'HitRate@10': data.get('hitrate@10', 0),
                'MRR': data.get('mrr', 0)
            })

df = pd.DataFrame(results)
print(f'Loaded {len(df)} results')
df.head(10)

## Step 6: Visualization - Model Comparison

In [ ]:
# Set style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 6)

# Plot 1: Recall@10 by Model Type and Protocol
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, protocol in enumerate(['cold', 'full', 'loo100']):
    ax = axes[idx]
    subset = df[df['Protocol'] == protocol].sort_values('Recall@10', ascending=False)
    
    if len(subset) == 0:
        ax.set_title(f'Protocol: {protocol.upper()} (No data)')
        continue
    
    colors = {'CF': '#4CAF50', 'CB': '#2196F3', 'Hybrid': '#FF9800'}
    bar_colors = [colors.get(t, '#999999') for t in subset['Type']]
    
    bars = ax.barh(subset['Model'], subset['Recall@10'], color=bar_colors)
    ax.set_xlabel('Recall@10')
    ax.set_title(f'Protocol: {protocol.upper()}')
    ax.invert_yaxis()

plt.suptitle('Model Performance Comparison by Protocol', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Graph Ablation (G1 vs G2 vs G3)
fig, ax = plt.subplots(figsize=(12, 6))

# Filter to CF models on cold protocol for cleaner comparison
cf_cold = df[(df['Type'] == 'CF') & (df['Protocol'] == 'cold')]
if len(cf_cold) > 0:
    pivot = cf_cold.pivot_table(index='Model', columns='Graph', values='Recall@10', aggfunc='mean')
    pivot.plot(kind='bar', ax=ax, width=0.8)
    ax.set_ylabel('Recall@10')
    ax.set_xlabel('Model')
    ax.set_title('Graph Structure Ablation (Cold Protocol)', fontsize=14, fontweight='bold')
    ax.legend(title='Graph Type')
    plt.xticks(rotation=45, ha='right')
else:
    ax.set_title('No CF Cold data available')

plt.tight_layout()
plt.savefig('results/graph_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: CF vs CB vs Hybrid Comparison
fig, ax = plt.subplots(figsize=(10, 6))

if len(df) > 0:
    type_avg = df.groupby(['Type', 'Protocol'])['Recall@10'].mean().unstack()
    type_avg.plot(kind='bar', ax=ax, width=0.8)
    ax.set_ylabel('Average Recall@10')
    ax.set_xlabel('Model Type')
    ax.set_title('Average Performance: CF vs CB vs Hybrid', fontsize=14, fontweight='bold')
    ax.legend(title='Protocol')
    plt.xticks(rotation=0)
else:
    ax.set_title('No data available')

plt.tight_layout()
plt.savefig('results/type_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Summary Table

In [ ]:
# Create summary table - Best models per category
print('=' * 60)
print('TOP MODELS BY PROTOCOL')
print('=' * 60)

for protocol in ['cold', 'full', 'loo100']:
    print(f'\n--- {protocol.upper()} ---')
    subset = df[df['Protocol'] == protocol].sort_values('Recall@10', ascending=False)
    if len(subset) > 0:
        print(subset[['Type', 'Model', 'Graph', 'Recall@10', 'NDCG@10']].head(5).to_string(index=False))
    else:
        print('No data')

In [ ]:
# Export full results to CSV
if len(df) > 0:
    df.to_csv('results/ablation_full_results.csv', index=False)
    print('Results exported to results/ablation_full_results.csv')
    
    # Display full sorted table
    print('\nFull Results (sorted by Recall@10):')
    display(df.sort_values('Recall@10', ascending=False).head(20))
else:
    print('No results to export')

## Step 8: Statistical Analysis

In [ ]:
# Improvement analysis: G2 vs G1, G3 vs G1
print('=' * 60)
print('GRAPH IMPROVEMENT ANALYSIS')
print('=' * 60)

if len(df) > 0:
    for model in df['Model'].unique():
        model_data = df[df['Model'] == model]
        g1 = model_data[model_data['Graph'] == 'strict_g1']['Recall@10'].mean()
        g2 = model_data[model_data['Graph'] == 'strict_g2']['Recall@10'].mean()
        g3 = model_data[model_data['Graph'] == 'strict_g3']['Recall@10'].mean()
        
        if g1 > 0 and not pd.isna(g1):
            g2_imp = ((g2 - g1) / g1) * 100 if (g2 > 0 and not pd.isna(g2)) else 0
            g3_imp = ((g3 - g1) / g1) * 100 if (g3 > 0 and not pd.isna(g3)) else 0
            print(f'{model:15} | G1: {g1:.4f} | G2: {g2:.4f} ({g2_imp:+.1f}%) | G3: {g3:.4f} ({g3_imp:+.1f}%)')
else:
    print('No data available')